## This script simulates the response curve of duplexes under different conditions

In [19]:
import numpy as np
import altair as alt
import pandas as pd

from biophysical_models import (calc_rsp, calc_rsp_deriv)
from fitting import params_from_array

1. Experiment settings

In [20]:
conc_ds               = 2.5e-6   # strand concentration (M) for bimolecular models; ignored if intramolecular
is_intramolecular     = False    # True for hairpins (unimolecular)
is_self_complementary = False    # True for self-complementary duplexes (homodimer)

# Temperature range of the simulation in Kelvin
temp = np.arange(0, 101, 1) + 273.15

2. Setup initial parameters

In [21]:
delta_h = -767e3           # Change in enthalpy in J/mol
delta_s = -2136            # Change in entropy in J/(mol·K)
b_ds    = 0.77             # Y-intercept of the linear baseline of the folded state
m_ds    = 0.00067          # slope of the linear baseline of the folded state
b_ss    = 1.43             # Y-intercept of the linear baseline of the unfolded state
m_ss    = -0.00032         # slope of the linear baseline of the unfolded state
params = np.asarray([ delta_h, delta_s, b_ds, m_ds, b_ss, m_ss], dtype=float)
params = params_from_array(params)

3. Calculate the response

**Equations**

Response with a linear baseline

$$Response = b_{ds} + m_{ds}T + (1 - \theta(T))\left[(b_{ss} - b_{ds}) + (m_{ss} - m_{ds})T\right]$$

Derivative with a linear baseline
$$\frac{dRsp}{dT} = m_{ds} + (m_{ss} - m_{ds})(1 - \theta(T)) - \big[(b_{ss} - b_{ds}) + (m_{ss} - m_{ds})T\big]
\left(\frac{\Delta H}{RT^2}\frac{\theta(T)}{\sqrt{1 + 4K_a(T) C_0}}
\right)$$

In [22]:
response = np.array(calc_rsp(temp, conc_ds, params, is_intramolecular, is_self_complementary))
response_df = pd.DataFrame({"temp": temp,
                   "response": response})

response_deriv = np.array(calc_rsp_deriv(temp, conc_ds, params, is_intramolecular, is_self_complementary))
response_derive_df = pd.DataFrame({"temp": temp,
                   "response": response_deriv})

4. Plot the simulated response curve

In [23]:
chart_direct = alt.Chart(response_df).mark_point().encode(
    x=alt.X('temp', scale=alt.Scale(domain=[273, 373]), title="Temperature (K)"),
    y=alt.Y('response', scale=alt.Scale(zero=False), title="Response"),
    tooltip=['temp', 'response'],
).properties(
    height=500,
    width=500,
    title='Simulated Response',
)

chart_direct

alt.Chart(...)

5. Plot the simulated derivative plot

In [24]:
chart_deriv = alt.Chart(response_derive_df).mark_point().encode(
    x=alt.X('temp', scale=alt.Scale(domain=[273, 373]), title="Temperature (K)"),
    y=alt.Y('response', scale=alt.Scale(zero=False), title="Response"),
    tooltip=['temp', 'response'],
).properties(
    height=500,
    width=500,
    title='Simulated Derivative',
)

chart_deriv

alt.HConcatChart(...)